# What does the forest use?

Full-centroid target, one-stage forest, one-minute relative-error tuning. Each example is held out in turn. The exact saved models are reconstructed and their predictions verified before interpretation. No predictor selection or resistance fits are performed by this notebook.

**Permutation:** 20 shuffles within each held-out network. Positive values mean that disrupting a variable worsens predictions. Related variables are shuffled jointly with the same row permutation. Scores are averaged equally across networks; the R² decrease uses the fixed global weighted variance. This is not the average of individual network R² values. Group importance is not the sum of individual importances.

**SHAP:** exact, tree-path-dependent TreeSHAP on the held-out predictions. Values are in minutes relative to each model's training-path expectation. Negative values move predictions toward advances; positive values move them toward delays. Absolute SHAP measures how much the model uses a variable, not whether using it helps prediction. These are neither causal effects nor a refitted variable-removal experiment. [TreeExplainer documentation](https://shap.readthedocs.io/en/latest/generated/shap.TreeExplainer.html).

Within-network permutation cannot assess information that is constant within that network. Both methods can divide importance among correlated variables. The interaction plots show how the fitted model behaves, not new physical laws.

## Model-version note — 19 October 2026

This notebook explains the earlier relative-sweep forests, **not the final tuned seven-variable frozen forest**. Exposure was subsequently removed. Preserve these results with their original provenance; fresh final-model SHAP/permutation analysis remains pending. See the [recorded follow-up investigations and statistical-model discussion](hydrograph_frozen_residual_investigation.ipynb) for completed entropy and residual-feature checks.

## Findings from the current run

The forest is driven primarily by **mean-width ratio** and **parent domain mean travelled length**, followed by the **mean travelled-length ratio**. Their average absolute held-out SHAP contributions are 6.97, 5.30 and 0.67 minutes respectively. Width ratio and parent domain size worsen both loss metrics when permuted in every held-out network; mean-length ratio does so in 14 of 15 networks.

Larger width ratios consistently push predictions toward advances; higher length ratios toward delays (or smaller advances). Larger parent domains tend to amplify advances in this dataset. The width × size interaction is strong, with mean absolute interaction allocation 7.01 minutes; the length-ratio × size interaction is 0.45 minutes. These are properties of the fitted forests, not causal estimates.

Route entropy has a smaller contribution and overlaps strongly with width ratio and mean-length ratio. Length SD, width CV and dominance have small contributions. Exposure is effectively 1 everywhere and has zero permutation/SHAP importance. This supports a later reduced-panel comparison, not immediate removal based on importance alone.

**Correlation matters:** width ratio and resistance have pooled Spearman correlation 0.861; width ratio and entropy −0.857. Joint shuffling of length and width produces less damage than shuffling width alone, consistent with individual shuffling creating inconsistent predictor combinations. Do not read the permutation losses as independent, additive physical contributions.

The resistance comparison is complete (58/58 checkpoints). Adding resistance improves centroid R² from 0.580 to 0.593, but worsens floored relative error from 71.6% to 72.6%. Replacing mean width gives R² 0.600 but relative error 79.4%. Keep mean width; resistance remains an optional addition rather than a clear improvement under the chosen relative-error objective.

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
ROOT=Path.cwd()
while not (ROOT/'RAPID').exists():
    if ROOT==ROOT.parent: raise RuntimeError('Run from inside river-hierarchy')
    ROOT=ROOT.parent
OUT=ROOT/'RAPID/outputs/hydrograph_forest_interpretation_20261019'
p=pd.read_csv(OUT/'permutation_summary.csv')
s=pd.read_csv(OUT/'shap_summary.csv')
shap=pd.read_parquet(OUT/'shap_values.parquet')
interactions=pd.read_parquet(OUT/'shap_interactions.parquet')
by=pd.read_csv(OUT/'permutation_by_network.csv',dtype={'example':str})
print('Individual predictors; equal-network means')
display(p[~p.group.str.endswith('_group')].sort_values('relative_increase_pp',ascending=False))
display(s)
print('Joint permutations of related predictors')
display(p[p.group.str.endswith('_group')].sort_values('relative_increase_pp',ascending=False))

Individual predictors; equal-network means
Joint permutations of related predictors


,group,relative_increase_pp,mse_increase_minutes2,networks_helped_relative,networks_helped_mse,r2_decrease
2,R_W,182.880564,136.168534,15,15,1.034995
8,muL_Dp / 1 km,75.099581,85.433557,15,15,0.649366
3,R_muL,24.732473,5.069328,14,14,0.038531
1,R_H,5.289806,0.167599,11,12,0.001274
4,R_sigmaL,2.380298,0.536294,12,8,0.004076
0,R_CVW,1.082176,0.166132,10,12,0.001263
5,d_Dp,0.407531,-0.060832,9,7,-0.000462
9,r_in,0.000000,0.000000,0,0,0.000000


,feature,symbol,mean_abs_shap_minutes
0,log1p__B_ratio_width_mean,R_W,6.970088
1,domain_size,muL_Dp / 1 km,5.297131
2,log1p__B_ratio_route_length_mean,R_muL,0.672716
3,log1p__B_ratio_route_entropy,R_H,0.439113
4,log1p__B_ratio_route_length_sd,R_sigmaL,0.122203
5,B_parent_dominance_mean,d_Dp,0.107188
6,log1p__B_ratio_width_cv,R_CVW,0.096831
7,B_exposure,r_in,0.000000


,group,relative_increase_pp,mse_increase_minutes2,networks_helped_relative,networks_helped_mse,r2_decrease
10,width_entropy_group,192.984722,141.479657,15,15,1.075363
11,width_group,185.361959,136.962410,15,15,1.041029
7,length_width_group,137.961508,135.732212,15,15,1.031678
6,length_group,87.464235,77.334678,15,15,0.587808


## Symbols and domains

All D values use the agreed outer comparison domain. R denotes child/parent. Model inputs use ln(1+R), while SHAP dependence plots below display the untransformed ratios.

| Symbol | Quantity |
|---|---|
| R_muL | Mean travelled-length ratio |
| muL_Dp / 1 km | Parent domain mean travelled length, divided by 1 km |
| R_sigmaL | Route-length SD ratio |
| R_W | Length-weighted mean-width ratio |
| R_CVW | Width-CV ratio |
| R_H | Route-entropy ratio |
| d_Dp | Parent mean split dominance |
| r_in | Domain exposure |

The grouped permutations cover length (ratio, SD ratio, parent size), width (mean ratio, CV ratio, dominance), length plus width ratios, and width plus entropy. The matrix below helps identify shared information. Exposure differences at machine precision are listed explicitly; the original preprocessing has been retained to reproduce the evaluated model.

In [5]:
fig=px.bar(p[~p.group.str.endswith('_group')].sort_values('relative_increase_pp'),x='relative_increase_pp',y='group',orientation='h',labels={'relative_increase_pp':'Increase in floored relative error (percentage points)','group':'Predictor'})
fig.show()
fig=px.imshow(by.pivot(index='example',columns='group',values='delta_relative_pp'),color_continuous_scale='RdBu_r',color_continuous_midpoint=0,labels={'color':'Relative-error increase (pp)'})
fig.show()
corr=pd.read_csv(OUT/'predictor_spearman.csv',index_col=0)
px.imshow(corr,color_continuous_scale='RdBu_r',zmin=-1,zmax=1,title='Pooled Spearman correlations; descriptive').show()
support=pd.read_csv(OUT/'predictor_support.csv',dtype={'example':str})
display(support.groupby(['symbol','numerically_constant']).size().rename('network_count').reset_index())

,symbol,numerically_constant,network_count
0,R_CVW,False,15
1,R_H,False,15
2,R_W,False,15
3,R_muL,False,15
4,R_sigmaL,False,15
5,d_Dp,False,15
6,muL_Dp / 1 km,False,15
7,r_in,True,15


## SHAP direction and size interactions

Each point is explained by the forest that did not train on its network. Plots show every comparison, so dense networks occupy more visual space; the summary table above gives each network equal weight. Fold-specific baselines differ, so these plots should not be read as one universal response curve. The pair interaction shown is twice the off-diagonal SHAP interaction entry.

In [7]:
size=shap[shap.symbol=='muL_Dp / 1 km'][['example','state','value']].rename(columns={'value':'parent_size_km'})
for symbol in ['R_muL','R_W','R_H','muL_Dp / 1 km']:
    a=shap[shap.symbol==symbol].merge(size,on=['example','state'],validate='one_to_one')
    px.scatter(a,x='value',y='shap_minutes',color='parent_size_km',hover_data=['example','state','edit_type'],title=symbol,labels={'value':symbol,'shap_minutes':'SHAP contribution (minutes)'}).show()
for pair,a in interactions.groupby('pair'):
    px.scatter(a,x='ratio',y='interaction_minutes',color='domain_size',hover_data=['example','state'],title=pair).show()
print('Extreme compound: attribution of its predicted shift, not of its prediction error')
a=shap[(shap.example=='37')&(shap.state=='CB001_bubble_3')]
display(a[['symbol','value','shap_minutes','base_minutes','predicted_minutes']])

Extreme compound: attribution of its predicted shift, not of its prediction error


,symbol,value,shap_minutes,base_minutes,predicted_minutes
7136,R_muL,0.825574,-4.223631,-5.945316,-85.02111
7531,muL_Dp / 1 km,89.967244,-43.314321,-5.945316,-85.02111
7926,R_sigmaL,0.000000,-0.127801,-5.945316,-85.02111
8321,R_W,7.452605,-30.862500,-5.945316,-85.02111
8716,R_CVW,0.000000,0.080190,-5.945316,-85.02111
9111,R_H,0.000000,-0.515141,-5.945316,-85.02111
9506,d_Dp,0.723589,-0.112590,-5.945316,-85.02111
9901,r_in,1.000000,0.000000,-5.945316,-85.02111


## Focused resistance comparison

Three panels: current predictors; current + −log10(λK); and resistance replacing mean-width ratio. Resistance uses the existing outer-domain definition. The two new panels select their forest settings independently inside the same grouped inner folds, using the same one-minute relative-error objective. The baseline reuses the completed forest predictions. Both all-example and without-54 scenarios are planned.

The comparison is complete: all 58 checkpoints are present. A result marked `complete=False` must not be compared with a complete panel. Peak scores reuse centroid predictions and are not separate peak-trained models.

In [9]:
scores=pd.read_csv(OUT/'resistance_scores.csv')
display(scores[(scores.scenario=='all')&(~scores.exclude_extreme)&(scores.subset=='all')])
print('Planned fits:',pd.read_csv(OUT/'resistance_run_plan.csv').fits.sum())

scenario,panel,response,exclude_extreme,subset,n,complete,relative_error_pct,mae,rmse,r2
all,current,centroid_shift_hours,False,all,2170,True,71.609316,2.937865,7.433075,0.580049
all,current,peak_shift_hours,False,all,2170,True,79.394936,4.502415,11.158720,0.495511
all,plus_resistance,centroid_shift_hours,False,all,2170,True,72.598555,2.877361,7.316803,0.593085
all,plus_resistance,peak_shift_hours,False,all,2170,True,79.440829,4.431503,11.151942,0.496124
all,resistance_replaces_width,centroid_shift_hours,False,all,2170,True,79.411977,2.955522,7.250112,0.600469
all,resistance_replaces_width,peak_shift_hours,False,all,2170,True,85.001992,4.629975,11.486360,0.465450


Planned fits: 1102
